# AI tools in Danish SMEs

Exam assignment, Applied Quantitative Methods 2026-27

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/exam/exam_assignment.ipynb)

This notebook is the assignment for the exam in Applied Quantitative Methods. Your group answers the questions in
this notebook and hands it in as the prerequisite for the oral exam. At the exam you present the notebook, discuss
the choices you made, and answer an individual question from the curriculum. No aids are allowed at the oral exam,
so make sure that every member of the group can explain each step and the statistics behind it.

All groups work on the same data and the same questions. The data are synthetic: the agency, the firms and all
numbers are invented for the course, and the results say nothing about real Danish firms.

## How to work with this notebook

- Save a copy in your Google Drive (File, Save a copy in Drive) before you start, and share it within the group.
- Answer each question in the cells under it: code in the code cell, and your written answer in the markdown cell
  marked *Your answer*. Add more cells if you need them, but keep the numbering of the questions.
- Write every answer in words a manager can follow, and keep it short and to the point: most answers need two to
  five sentences, and none should need more than about 150 words. Where a question expects a longer answer, the word
  limit is given in the question. Report the numbers you base an answer on, rounded sensibly, and do not repeat what
  the code output already shows.
- Each section names the lectures that cover it. Sections 1 to 4 can be done right away, section 5 after
  Lectures 6 to 8, and section 6 after Lecture 9. Lecture 5 works through a practice case with the same kind of steps
  as sections 1 to 4, and Lecture 10 does the same for sections 5 to 7.
- You are welcome to use an LLM while you work. Say briefly in section 0 how you used it. You are responsible for
  every line of code and every claim in the notebook, and you must be able to explain them without help at the exam.
- Show the full output of `.summary()` for every regression model you estimate in sections 5 and 6, and keep it in
  the notebook you hand in. At the exam you will be asked to read and explain this output.
- Before you hand in, restart the runtime and run all cells from the top (Runtime, Restart and run all), so the
  notebook runs without errors and shows all outputs.

Each group hands in one notebook (.ipynb) in Digital Eksamen by 13 January 2027.

## 0 Use of LLMs

Write two or three sentences on how you used an LLM in your work on this notebook, if you did.

*Use of LLMs:*

## 1 The case

*Covered in Lectures 1 to 5.*

The Kattegat Business Agency is a regional business development agency that advises small and medium-sized
enterprises (SMEs, 1 to 249 employees) across Denmark. In early 2024 the agency drew a random sample of SMEs from
the Central Business Register (CVR) and invited them to take part in a study that would follow the firms for three
years. Of the invited firms, 2,000 agreed. When they joined, the agency recorded the firm's sector, region, age and
number of full-time employees, whether the manager has a university degree, and the firm's digital maturity on the
agency's own scale from 1 to 10. The agency's strategy has a target that the average digital maturity of SMEs is at
least 5 on this scale.

Later in 2024, 300 of the 2,000 firms, drawn by lot, took part in a free course from the agency on AI tools in daily
work. The other 1,700 firms did not.

In spring 2025 the agency asked all firms in the study whether they use AI tools, such as chatbots, text and image
generators, and tools for forecasting or bookkeeping, in their daily work. In 2026 it linked the answers to the
firms' annual accounts for 2023 and 2025.

The agency asks four questions:

- A. How widespread is the use of AI tools among SMEs, and does the average digital maturity reach the agency's
  target of 5 on its 1 to 10 scale?
- B. Did the course work? Did it make firms use AI tools, and did it raise their productivity?
- C. Is the use of AI tools associated with higher productivity, and is the association the same in
  knowledge-intensive sectors (business services and ICT) as in other sectors?
- D. Which firms use AI tools?

### The data

The file `sme_ai_survey.csv` holds the agency's data on the firms in the study, with one row per firm. The data are as
the agency received them, which means they have not been checked or cleaned.

| Variable | Content |
|---|---|
| `firm_id` | Firm ID |
| `region` | Region of the firm's address (the five Danish regions) |
| `sector` | Sector: Manufacturing, Construction, Retail and wholesale, Hospitality, Business services, ICT |
| `employees` | Number of full-time employees when the firm joined the study in 2024 |
| `firm_age` | Age of the firm in years when it joined the study in 2024 |
| `manager_degree` | Whether the manager has a university degree (Yes or No), recorded in 2024 |
| `digital_maturity` | The agency's index of digital maturity, from 1 (low) to 10 (high), recorded in 2024 before the course; 99 means "don't know" |
| `course` | 1 if the firm was drawn by lot and took the AI course in 2024, 0 otherwise |
| `ai_adopted` | Whether the firm uses at least one AI tool in its daily work, from the survey in spring 2025 (Yes or No) |
| `productivity_2023` | Revenue per full-time employee in 2023, in thousand DKK |
| `productivity_2025` | Revenue per full-time employee in 2025, in thousand DKK |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/sme_ai_survey.csv"
raw = pd.read_csv(url)
raw.head()

### 1.1

For each of the agency's four questions, say in your own words what it asks about in statistical terms (for example
a share, a mean, a difference between groups, or an association) and for which firms. Some questions have more than
one part. Name the variables in the data that you will use to answer each question. (At most 250 words for 1.1 and
1.2 together.)

*Your answer:*

### 1.2

Answer three short questions about what the data can show. Think about how the firms were selected, how adoption
was measured, which comparisons in the data are based on a random draw, and when each variable was measured.

(a) Which population do the data describe?

(b) Give one thing the data can tell the agency, and explain why.

(c) Give one thing the data cannot tell the agency, and explain why.

*Your answer:*

## 2 Data preparation

*Covered in Lecture 5.*

Keep the raw data in `raw` and do your cleaning on a copy, so you can always compare with the original. Every decision
you make here affects the results later, and you will be asked about it at the exam.

### 2.1

Inspect the data: the number of rows and columns, the data type of each column, and the values that occur in the
text columns. Describe the problems you find.

*Your answer:*

### 2.2

Clean the data in three steps. For each problem you find, count the rows affected and record the problem in your
cleaning log before you fix it (see 2.3). Do every fix in code. Some problems can be handled in more than one
defensible way; name the choice you make and justify it.

#### 2.2 (a)

Fix the problems that have one correct solution, such as duplicate rows, inconsistent category labels and numbers
stored as text.

*Your answer:*

#### 2.2 (b)

Handle the values that cannot be right, such as codes that stand for missing values and impossible values. For each,
say what you did and why.

*Your answer:*

#### 2.2 (c)

Look for values that are possible but suspicious, such as values far from the rest of the data. Explain how you
told a value that is an error from a value that is unusual but real, and what you did with the errors.

*Your answer:*

### 2.3

Show your cleaning log as a table with one row per problem: the problem, the number of rows affected (counted in
code before you fixed it), your decision, and the consequence of that decision for the analysis. For example, what
would have happened to the mean of a variable if you had not handled a problem? The note on the cleaning log at the
top of the Workshop 5 notebook shows how to count rows with conditions.

*Your answer:*

### 2.4

Examine the missing values after cleaning: how many are there in each variable, and is a value missing more often
for some kinds of firms than for others? What does that mean for the results, and how do you
handle the missing values in the rest of the notebook?

*Your answer:*

### 2.5

Create the variables you need later: `ai` (1 if the firm uses AI tools, 0 otherwise), `degree` (1 if the manager has
a university degree), `knowledge` (1 for business services and ICT), the natural logarithm of `employees` and of the
two productivity variables, and the change in productivity from 2023 to 2025 in thousand DKK and in percent
(100 · (productivity_2025 − productivity_2023) / productivity_2023). Check that the new variables look as you expect,
and that productivity in 2023 and 2025 are on the same scale.

*Your answer:*

## 3 Exploratory analysis

*Covered in Lectures 1 and 5.*

The point of this section is to get to know the data before you test and model. Aim the analysis at the questions in
sections 4 to 6.

### 3.1

Make a table of descriptive statistics for the numerical variables (count, mean, median, standard deviation,
minimum, maximum). Compare the mean and the median of `employees` and of the productivity variables. What does the
comparison tell you about the shape of the distributions?

*Your answer:*

### 3.2

Plot the distribution of `employees` and of `productivity_2025`, each on the original scale and after taking logs,
so that you have one plot for each variable on each scale.
Which scale would you use in a regression, and why?

*Your answer:*

### 3.3

Show the share of firms using AI tools by sector and by region, and compare productivity in 2025 between firms that
use AI tools and firms that do not. Use tables or plots. What patterns do you see, and which of them do you expect
to test or model later?

*Your answer:*

### 3.4

Compute the correlation matrix of the numerical variables (a heatmap is optional). Are any of the explanatory variables you plan to use in
section 5 strongly correlated with each other, and what would that mean for the regressions?

*Your answer:*

## 4 Estimation and tests

*Covered in Lectures 2 to 4.*

For each test, write the hypotheses H<sub>0</sub> and H<sub>1</sub> in symbols and in words, say whether the test is
one-sided or two-sided and why, choose the test and argue for it (paired or independent samples; for two means,
pooled or Welch), check the conditions, compute the result, and give the answer in words for the agency. Use a
significance level of 5% unless the question says otherwise. Each of these steps can be short, one or two lines.

### 4.1

Estimate the share of SMEs that use AI tools, with a 95% confidence interval. Use only the firms that did not
take the course, and explain why. Explain what the interval means.

*Your answer:*

### 4.2

The agency's target is an average digital maturity of at least 5. Do the data show that the mean digital maturity
of SMEs is above 5?

*Your answer:*

### 4.3

Compare the share of firms using AI tools in 2025 between the firms that took the course and the firms that did
not. Did the course make more firms use AI tools? Explain why the draw by lot allows the agency to read the
difference as an effect of the course.

*Your answer:*

### 4.4

Compare the change in productivity in percent from 2023 to 2025 between the firms that took the course and the
firms that did not, and give a 95% confidence interval for the difference (in percentage points). What can the
agency conclude, and what can it not conclude, from the result?

*Your answer:*

### 4.5

Among the firms that use AI tools, did productivity change from 2023 to 2025? Explain why the test you use fits these
data, and why this comparison alone does not show that AI tools raised productivity.

*Your answer:*

### 4.6

Does the variability of the change in productivity in percent differ between hospitality and manufacturing? Test
whether the two variances differ (a two-sided test), and say what a difference in variability would mean for a firm in
each sector.

*Your answer:*

## 5 Regression

*Covered in Lectures 6 to 8.*

In this section the dependent variable is the natural logarithm of productivity in 2025. For each model, interpret
the coefficients you are asked about in words. With log productivity as the dependent variable, a coefficient b on a
dummy variable means a difference of about 100·b percent when b is small; the exact difference is
100·(e<sup>b</sup> − 1) percent. Show the full `.summary()` output for each model. If a model includes sector as a
categorical variable, `C(sector)`, do not also include `knowledge`, since it is already determined by the sector
dummies.

### 5.1

Estimate a simple regression of log productivity in 2025 on `ai`. Interpret the coefficient on `ai`.

*Your answer:*

### 5.2

Estimate a multiple regression that adds control variables. Choose the variables from the data and argue for each
one, and say which categories you use as reference categories for `sector` and `region`. Do not include
productivity in 2023 yet. How does the coefficient on
`ai` change compared with 5.1? Explain why.

*Your answer:*

### 5.3

Why use log productivity as the dependent variable? Estimate the model from 5.2 twice, once with productivity in 2025
on the original scale and once in logs, and test both for heteroskedasticity. Compare the residual plots and the
tests, and explain the difference.

*Your answer:*

### 5.4

Add log productivity in 2023 to the model from 5.2. What does the coefficient on `ai` measure now? Compare the
coefficient and R<sup>2</sup> with 5.2.

*Your answer:*

### 5.5

Does the association between AI tools and productivity differ between knowledge-intensive sectors and other sectors?
Add the interaction between `ai` and `knowledge` to the model from 5.4 and interpret the result. (With `C(sector)` in
the model, include only the interaction term `ai:knowledge`.)

*Your answer:*

### 5.6

Check the assumptions behind your model from 5.5: plot the residuals against the fitted values and by sector, test
for heteroskedasticity, check whether the residuals are approximately normal, and compute the variance inflation
factors. Which assumptions hold, and which do not? Can you relate what you find to a result in section 4?

*Your answer:*

### 5.7

For each assumption that fails in 5.6, explain what it means for the coefficients, the standard errors and the tests,
and what you do about it. Show whether your conclusions from 5.5 change.

*Your answer:*

### 5.8

Use your model from 5.5 to predict the productivity in 2025 of an ICT firm in Nordjylland with 12 employees, a
firm age of 15 years, a digital maturity of 6, a manager with a university degree, a productivity of 900 thousand DKK
in 2023, and which uses AI tools. If your model includes other variables, set them to values you choose and say
which. Give the prediction in thousand DKK (taking the exponential of the predicted log value is enough here).
Then predict the same firm without AI tools and compare.

*Your answer:*

### 5.9

Can the agency conclude from sections 5.1 to 5.5 that AI tools raise productivity? Discuss which other explanations
could produce the same results, and compare with what section 4 could show about the course.

*Your answer:*

## 6 Which firms use AI tools?

*Covered in Lectures 8 and 9.*

In this section the dependent variable is `ai`. Use log employees, digital maturity, the manager's degree, the
course, sector and region as explanatory variables. Show the full `.summary()` output for each model.

### 6.1

Estimate a linear probability model. Interpret the coefficients on digital maturity and on the course.
How many fitted values lie outside the interval from 0 to 1, and why is that a problem?

*Your answer:*

### 6.2

Estimate a logit or a probit model with the same variables. Compute the average marginal effects and compare them
with the coefficients of the linear probability model.

*Your answer:*

### 6.3

Use the model from 6.2 to predict the probability that a construction firm in Sjælland with 8 employees, a digital
maturity of 4 and a manager without a university degree uses AI tools, with and without the course.

*Your answer:*

### 6.4

Which of the two models would you report to the agency, and why?

*Your answer:*

## 7 Conclusions

### 7.1

Write a summary for the agency's management of at most 300 words that answers questions A to D in section 1. Write
for readers with no training in statistics: no symbols and no p-values, but say how certain the results are.

*Your answer:*

### 7.2

Describe the main limitations of your analysis: the quality of the data (see section 2), how productivity is
measured and what besides the firm's efficiency can change revenue per employee, what the results can and cannot say
about cause and effect, important variables that are missing from the data, and to which firms the
results can be generalized. (At most 250 words.)

*Your answer:*